In [1]:
using JuMP
using HiGHS
using LinearAlgebra

model = Model(HiGHS.Optimizer)
set_silent(model)

@variable(model, x[1:3] >= 0)
@variable(model, sp[1:2] >= 0)
@variable(model, sm[1:2] >= 0)

@constraint(model, 4x[1] - x[2] + 7x[3] - 100 == sp[1] - sm[1])
@constraint(model, 9x[1] - 10x[2] + x[3] - 150 == sp[2] - sm[2])

@objective(model, Min, sp[1] + sp[2] + sm[1] + sm[2])

optimize!(model)

Since there are only two constarints with three variables, if there is a solution, there may be more than one.  If there is, find possible ranges for the variables.

In [2]:
# 1. Solve the initial model+
optimize!(model)

if termination_status(model) == MOI.OPTIMAL
    opt_val = objective_value(model)  # Expecting 0.0

    # 2. Add a constraint fixing the objective to its optimal value
    @constraint(model, opt_objective, sp[1] + sp[2] + sm[1] + sm[2] == opt_val)

    # 3. Query min and max for each x variable across the optimal space
    for i ∈ 1:3
        @objective(model, Min, x[i])
        optimize!(model)
        min_val = value(x[i])

        @objective(model, Max, x[i])
        optimize!(model)
        max_val = value(x[i])

        println("x[$i] optimal range: [$(round(min_val, digits=4)), $(round(max_val, digits=4))]")
    end
else
    println("The initial model did not solve to optimality.")
end

x[1] optimal range: [16.1017, 27.4194]
x[2] optimal range: [0.0, 9.6774]
x[3] optimal range: [0.0, 5.0847]
